# Sprint 1: Infraestructura de Datos y Pipeline de Calibracion

> **Prerrequisitos:** Solo requiere ejecutar el **Bloque 0** (configuracion de entorno y parches).
> Este sprint es el punto de entrada del proyecto.

**Objetivo:** Consolidar el entorno de datos local en WSL2 y construir el cargador de tensores
que alimentara los algoritmos de cuantizacion offline (GPTQ y AWQ) en la RTX 4060 Ti.

**Dataset:** [fsicoli/common_voice_17_0](https://huggingface.co/datasets/fsicoli/common_voice_17_0) (config `es`)
- Mirror comunitario completo de Mozilla Common Voice 17.0 Spanish (~2,220h total, 562h validadas)
- Splits nativos: `train`, `validation`, `test`, `other`, `invalidated`
- Audio original a 16kHz mono, licencia CC0-1.0
- **Motivo del cambio:** Mozilla retiro CV 17.0 de HuggingFace en Oct 2025. El mirror `fsicoli` almacena los audios directamente.

> **Nota tecnica:** VibeVoice 1.5B procesa audio crudo a **24kHz mono**, no espectrogramas de Mel.
> El preprocesamiento consiste en remuestreo + normalizacion de amplitud, no en conversion frecuencial.

## Orden de ejecucion: Sprint 1 y Sprint 1.5

Ejecute los identificadores en este orden. No avance a Sprints 2-7 hasta que `S1.5-07` apruebe el gate WER.

| Orden | Entorno | Accion |
|---|---|---|
| `S1-00` | Notebook | Entorno, rutas y parche CONFIG_MAPPING |
| `S1-01` | Notebook | Descargar Common Voice 17 ES |
| `S1-02` | Notebook | Exploracion opcional del dataset |
| `S1-03` | Notebook | Cargar y asignar splits nativos |
| `S1-04` | Notebook | Guardar trazabilidad de splits |
| `S1-05` | Notebook | Definir y validar preprocesamiento |
| `S1-06` | Notebook | Construir calibracion estratificada |
| `S1-07` | Notebook | Guardar tensor y metadatos |
| `S1-08` | Notebook | Verificar calibracion |
| `S1.5-01` | Notebook | Crear train/dev speaker-disjoint |
| `S1.5-02` | Notebook | Validar script de entrenamiento |
| `S1.5-03` | Terminal | `bash scripts/run_finetune_es.sh` |
| `S1.5-04` | Notebook | Validar script de merge |
| `S1.5-05` | Terminal | `bash scripts/merge_es_checkpoint.sh` |
| `S1.5-06` | Notebook | Cargar checkpoint BF16 corregido |
| `S1.5-07` | Notebook | Gate linguistico WER/CER y WAV |


---
## Bloque 0: Configuracion del Entorno y Parches Criticos

Se aplica el parche obligatorio de `transformers` para evitar colisiones de mapeo
con el fork comunitario de VibeVoice (ver `agents.md` seccion 3).

In [3]:
print("[S1-00] Configuracion del entorno y rutas")
import sys
import os
import json
import time
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import transformers
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm

# --- Parche critico: colision de mapeo en transformers ---
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)
print("Parche de CONFIG_MAPPING aplicado.")

# --- Rutas del proyecto ---
PROJECT_ROOT = Path(os.path.abspath("")).parent if "notebooks" in os.path.abspath("") else Path(os.path.abspath(""))
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"WEIGHTS_DIR  : {WEIGHTS_DIR}")
print(f"Torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Parche de CONFIG_MAPPING aplicado.
PROJECT_ROOT : /home/alfrog/projects/VibeVoice_Optimization
DATA_DIR     : /home/alfrog/projects/VibeVoice_Optimization/data
WEIGHTS_DIR  : /home/alfrog/projects/VibeVoice_Optimization/weights
Torch        : 2.12.0+cu130  |  CUDA: True
GPU          : NVIDIA GeForce RTX 4060 Ti
VRAM Total   : 8.00 GB


---
## TSK-1.1: Descarga Automatizada del Dataset

Se descarga el subset en espanol de Common Voice 17.0 desde el mirror comunitario
`fsicoli/common_voice_17_0` con config `"es"`. El dataset contiene las particiones
nativas `train`/`validation`/`test`/`other` con audios y transcripciones.

**Estrategia:** Cargamos el split `train` como base para el 70% de datos de
entrenamiento (fine-tuning), y los splits `test` y `validation` nativos como
nuestros sets de benchmark y calibracion respectivamente.

In [4]:
print("[S1-01] Descarga de Common Voice 17 espanol")
DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"

print(f"Descargando dataset: {DATASET_ID} (config={DATASET_CONFIG})")
print("(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)\n")

t_start = time.time()

cv17_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    trust_remote_code=True,
)

t_elapsed = time.time() - t_start
print(f"\nDescarga completada en {t_elapsed/60:.1f} minutos.")
print(f"Registros cargados (train): {len(cv17_dataset):,}")
print(f"Columnas: {cv17_dataset.column_names}")

Descargando dataset: fsicoli/common_voice_17_0 (config=es)
(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)



Reading metadata...: 336846it [00:01, 302211.76it/s]s/s]
Generating train split: 336846 examples [00:40, 8267.73 examples/s] 
Reading metadata...: 15857it [00:00, 328621.65it/s]xamples/s]
Generating validation split: 15857 examples [00:02, 7788.17 examples/s]
Reading metadata...: 15857it [00:00, 350204.45it/s]s/s]
Generating test split: 15857 examples [00:02, 7865.53 examples/s]
Reading metadata...: 1138631it [00:03, 311900.66it/s]/s]
Generating other split: 1138631 examples [02:04, 9145.43 examples/s] 
Reading metadata...: 86227it [00:00, 352991.03it/s]examples/s]
Generating invalidated split: 86227 examples [00:09, 8963.00 examples/s] 



Descarga completada en 28.6 minutos.
Registros cargados (train): 336,846
Columnas: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']


---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [5]:
print("[S1-02] Exploracion opcional del dataset")
# --- Muestra de registros ---
print("=== Ejemplos del dataset ===\n")
for i in range(3):
    sample = cv17_dataset[i]
    audio_info = sample["audio"]
    duration_s = len(audio_info["array"]) / audio_info["sampling_rate"]
    print(f"[{i}] \"{sample['sentence'][:80]}...\"")
    print(f"    Duracion: {duration_s:.2f}s | SR: {audio_info['sampling_rate']}Hz")
    print(f"    Locale: {sample['locale']} | Genero: {sample['gender']} | Edad: {sample['age']}")
    print()

# --- Distribucion de duraciones (muestreo estadistico) ---
SAMPLE_SIZE = 5000
indices = random.sample(range(len(cv17_dataset)), min(SAMPLE_SIZE, len(cv17_dataset)))
durations = []
for idx in tqdm(indices, desc="Muestreando duraciones"):
    arr = cv17_dataset[idx]["audio"]["array"]
    sr = cv17_dataset[idx]["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

durations = np.array(durations)
print(f"\n=== Estadisticas de Duracion (muestra n={len(durations)}) ===")
print(f"Media    : {durations.mean():.2f}s")
print(f"Mediana  : {np.median(durations):.2f}s")
print(f"Min/Max  : {durations.min():.2f}s / {durations.max():.2f}s")
print(f"Std      : {durations.std():.2f}s")
print(f"Horas est. totales: {durations.mean() * len(cv17_dataset) / 3600:.1f}h")

# --- Distribucion de genero y edad ---
genders = Counter()
ages = Counter()
for idx in tqdm(indices, desc="Muestreando demografia"):
    sample = cv17_dataset[idx]
    genders[sample["gender"]] += 1
    ages[sample["age"]] += 1

print(f"\nGeneros  : {dict(genders.most_common())}")
print(f"Edades    : {dict(ages.most_common())}")

=== Ejemplos del dataset ===

[0] "A ese grupo de amigos les gusta comer helados por la tarde...."
    Duracion: 3.89s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[1] "A mi profesora le importa demasiado la gramática y muy poco el contenido...."
    Duracion: 5.16s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[2] "A decir verdad , eres demasiado genial ...."
    Duracion: 3.05s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties



Muestreando duraciones: 100%|██████████| 5000/5000 [00:33<00:00, 148.61it/s]



=== Estadisticas de Duracion (muestra n=5000) ===
Media    : 5.19s
Mediana  : 5.11s
Min/Max  : 1.58s / 10.44s
Std      : 1.44s
Horas est. totales: 485.2h


Muestreando demografia: 100%|██████████| 5000/5000 [00:13<00:00, 360.70it/s]


Generos  : {'male_masculine': 2864, 'female_feminine': 1576, '': 560}
Edades    : {'twenties': 2387, 'thirties': 616, '': 542, 'sixties': 509, 'fifties': 389, 'teens': 292, 'fourties': 262, 'seventies': 1, 'nineties': 1, 'eighties': 1}


---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [6]:
print("[S1-03] Carga y asignacion de splits nativos")
# --- Cargar todos los splits nativos de CV17 es ---
print("Cargando splits nativos de Common Voice 17.0 (es)...")

cv17_all = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    trust_remote_code=True,
)

for split_name, split_data in cv17_all.items():
    print(f"  {split_name:<20} {len(split_data):>10,} registros")

train_dataset = cv17_all["train"]
test_dataset = cv17_all["test"]
val_dataset = cv17_all["validation"]
other_dataset = cv17_all.get("other", None)

print(f"\n=== Asignacion para el proyecto ===")
print(f"Train (fine-tuning)       : {len(train_dataset):,} registros")
print(f"Test (benchmark)          : {len(test_dataset):,} registros")
print(f"Validation (calibracion)  : {len(val_dataset):,} registros")
if other_dataset:
    print(f"Other (calibracion extra) : {len(other_dataset):,} registros")

Cargando splits nativos de Common Voice 17.0 (es)...
  train                   336,846 registros
  validation               15,857 registros
  test                     15,857 registros
  other                 1,138,631 registros
  invalidated              86,227 registros

=== Asignacion para el proyecto ===
Train (fine-tuning)       : 336,846 registros
Test (benchmark)          : 15,857 registros
Validation (calibracion)  : 15,857 registros
Other (calibracion extra) : 1,138,631 registros


In [7]:
print("[S1-04] Guardado de trazabilidad de splits")
# --- Guardar metadatos de particiones para trazabilidad ---
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_count": len(train_dataset),
    "test_count": len(test_dataset),
    "val_count": len(val_dataset),
    "other_count": len(other_dataset) if other_dataset else 0,
    "splits_used": {
        "fine_tuning": "train",
        "benchmark_test": "test",
        "calibration": "validation (+ other)",
    },
}

partitions_path = DATA_DIR / "dataset_partitions.json"
with open(partitions_path, "w") as f:
    json.dump(partitions_meta, f, indent=2)
print(f"Metadatos de particiones guardados en: {partitions_path}")

Metadatos de particiones guardados en: /home/alfrog/projects/VibeVoice_Optimization/data/dataset_partitions.json


---
## TSK-1.3: Preprocesamiento Acustico

VibeVoice 1.5B espera audio crudo monofonico a **24,000 Hz** (no espectrogramas de Mel).
El pipeline de preprocesamiento consiste en:

1. **Remuestreo:** 48,000 Hz → 24,000 Hz (usando `librosa.resample`)
2. **Conversion a mono:** garantizado por el dataset
3. **Normalizacion de amplitud:** a -25 dB FS (mismo criterio que `AudioNormalizer` del modelo)
4. **Validacion:** descarte de audios corruptos o de duracion minima (<0.5s)

El resultado se almacena como tensores `torch.float32` en archivos `.pt`.

In [8]:
print("[S1-05] Preprocesamiento de audio a 24 kHz y -25 dBFS")
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    """
    Preprocesa un array de audio para VibeVoice:
    remuestreo a 24kHz + normalizacion a target_dB_FS.
    Retorna None si el audio es invalido.
    """
    if audio_array is None or len(audio_array) == 0:
        return None

    # Remuestreo a 24kHz
    if orig_sr != TARGET_SR:
        audio = librosa.resample(
            y=audio_array.astype(np.float32),
            orig_sr=orig_sr,
            target_sr=TARGET_SR,
        )
    else:
        audio = audio_array.astype(np.float32)

    duration = len(audio) / TARGET_SR
    if duration < MIN_DURATION_S:
        return None

    # Normalizacion a target dB FS
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < EPS:
        return None
    scalar = (10 ** (TARGET_DB_FS / 20)) / (rms + EPS)
    audio = audio * scalar

    # Prevenir clipping
    peak = np.max(np.abs(audio))
    if peak > 1.0:
        audio = audio / peak

    return audio.astype(np.float32)

# --- Prueba con los primeros 5 registros ---
print("Validando preprocesamiento con 5 muestras...\n")
for i in range(5):
    sample = val_dataset[i]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    processed = preprocess_audio(arr, sr)
    if processed is not None:
        print(f"[{i}] OK: {len(processed)/TARGET_SR:.2f}s -> {processed.shape}, "
              f"peak={np.max(np.abs(processed)):.3f}, rms={np.sqrt(np.mean(processed**2)):.4f}")
    else:
        print(f"[{i}] DESCARTADO")

Validando preprocesamiento con 5 muestras...

[0] OK: 4.85s -> (116352,), peak=0.409, rms=0.0562
[1] OK: 5.57s -> (133632,), peak=0.453, rms=0.0562
[2] OK: 4.37s -> (104832,), peak=0.368, rms=0.0562
[3] OK: 5.18s -> (124416,), peak=0.358, rms=0.0562
[4] OK: 3.34s -> (80064,), peak=0.557, rms=0.0562


---
## TSK-1.4: Construccion del Set de Calibracion Offline

Se construye el tensor de calibracion a partir de la particion de validacion (10%).

**Estrategia:**
- Para GPTQ/AWQ se necesita un subconjunto representativo (tipicamente 128-512 muestras)
  que cubra diversidad de duraciones, generos y acentos.
- Se seleccionan `N_CALIB_SAMPLES` muestras con muestreo estratificado simple.
- Los audios preprocesados se concatenan en un unico tensor `[total_samples]`.
- Se guarda metadato con offsets para reconstruir limites de cada alocucion.

**Formato de salida:**
- `data/calibration_tensor.pt` — tensor float32 con audio concatenado
- `data/calibration_metadata.json` — indices originales, offsets, duraciones

In [9]:
print("[S1-06] Construccion de calibracion estratificada")
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Muestreo estratificado por acento y genero, con candidatos de reserva ---
calib_source = val_dataset
n_source = len(calib_source)
meta_columns = [name for name in ["accent", "gender"] if name in calib_source.column_names]
meta_view = calib_source.select_columns(meta_columns)
calib_groups = defaultdict(list)
for idx in range(n_source):
    row = meta_view[idx]
    key = tuple((row.get(name) or "unknown") for name in meta_columns)
    calib_groups[key].append(idx)
rng_calib = random.Random(CALIB_SEED)
for indices_group in calib_groups.values():
    rng_calib.shuffle(indices_group)
candidate_target = min(n_source, max(N_CALIB_SAMPLES, int(N_CALIB_SAMPLES * 1.25)))
calib_candidate_indices = []
active_groups = list(calib_groups)
rng_calib.shuffle(active_groups)
while active_groups and len(calib_candidate_indices) < candidate_target:
    next_groups = []
    for key in active_groups:
        if calib_groups[key]:
            calib_candidate_indices.append(calib_groups[key].pop())
        if calib_groups[key]:
            next_groups.append(key)
        if len(calib_candidate_indices) >= candidate_target:
            break
    active_groups = next_groups

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_candidate_indices, desc="Preprocesando calibracion"):
    if len(calib_audios) >= N_CALIB_SAMPLES:
        break
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
        "accent": sample.get("accent", ""),
        "client_id": sample.get("client_id", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")
if len(calib_audios) != N_CALIB_SAMPLES:
    raise RuntimeError(f"Calibracion incompleta: {len(calib_audios)}/{N_CALIB_SAMPLES}")

Seleccionando 512 muestras de calibracion...


Preprocesando calibracion: 100%|██████████| 512/512 [00:02<00:00, 190.64it/s]


Muestras procesadas : 512
Muestras rechazadas : 0
Total de muestras de audio : 74,889,543 (3120.4s)


In [10]:
print("[S1-07] Guardado del tensor de calibracion")
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

Tensor de calibracion : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_tensor.pt
Shape                 : torch.Size([74889543])
Tamaño en disco       : 285.7 MB
Metadatos             : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_metadata.json


In [11]:
print("[S1-08] Verificacion del set de calibracion")
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

=== Verificacion del Set de Calibracion ===

Tensor cargado  : torch.Size([74889543]), dtype=torch.float32
Duracion total  : 3120.4s
Numero de clips : 512

Primeros 3 clips:
  idx=     5 | 3.14s | "No lo he visto ...."
  idx=    27 | 5.14s | "Allá va el niño, donde le tratan con cariño...."
  idx=    97 | 6.82s | "de la mano , dando vueltas en setenta y cinco metros de eslo..."

Verificacion de offsets: OK


---
# Sprint 1.5: Fine-Tuning Monolingue Espanol con LoRA

> **Prerrequisitos:** Requiere haber ejecutado **Sprint 1 completo** (celdas Bloque 0 hasta TSK-1.4).
> Variables necesarias: `cv17_all`, `train_dataset`, `val_dataset`, `DATA_DIR`, `WEIGHTS_DIR`, `PROJECT_ROOT`.

**Objetivo corregido:** reparar la precision linguistica del checkpoint VibeVoice-ES
existente mediante un segundo LoRA CE-only. El resultado se escribe como
**VibeVoice-ES-Corrected** sin sobrescribir la linea base anterior.

**Estrategia:**
- LoRA rank=8 sobre proyecciones Q/K/V/O + gate/up/down del LLM (Qwen2.5)
- Diffusion head, connectors y tokenizadores congelados
- `voice_prompt_drop_rate=1.0`: no se generan prompts desde el propio target
- `ce_loss_weight=1.0` y `diffusion_loss_weight=0.0` para correccion linguistica
- Desarrollo speaker-disjoint; validation nativo sigue reservado para PTQ
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=64` para 8GB VRAM

> **Nota:** El entrenamiento ejecuta en terminal via `python -m vibevoice.finetune.train_vibevoice`.
> Las celdas de este bloque preparan los datos y generan el script de lanzamiento.
> La ejecucion del entrenamiento (~7-15h) se realiza fuera del notebook.

---
## TSK-1.5.1: Formatear Dataset al Esquema VibeVoice

El script de fine-tuning espera:
- Columna `text` con formato `"Speaker 1: {sentence}"`
- Columna `audio` como path a archivo de audio (str) o dict con array+sampling_rate
- Sin `voice_prompts`: la correccion es linguistica y usa drop rate 1.0

Guardamos los datos formateados como JSONL para pasarlos con `--train_jsonl` / `--validation_jsonl`.
Se crea un desarrollo speaker-disjoint a partir de `train` y esos hablantes se
excluyen del entrenamiento. El split nativo `validation` queda reservado para PTQ.
El texto se normaliza sin eliminar tildes, dieresis, `ñ`, `¿` ni `¡`.

In [ ]:
print("[S1.5-01] Creacion de train/dev speaker-disjoint")
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

def normalize_spanish_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text or ""))
    text = "".join(ch for ch in text if ch.isprintable())
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([.!?])\1+", r"\1", text)
    return text

# Desarrollo speaker-disjoint: ningun client_id de dev permanece en train.
N_VAL_TARGET = 2000
SPLIT_SEED = 42
speaker_indices = defaultdict(list)
for idx, client_id in enumerate(train_dataset["client_id"]):
    if client_id:
        speaker_indices[client_id].append(idx)
speakers = list(speaker_indices)
random.Random(SPLIT_SEED).shuffle(speakers)
val_indices_set = set()
val_speakers = []
for speaker in speakers:
    val_speakers.append(speaker)
    val_indices_set.update(speaker_indices[speaker])
    if len(val_indices_set) >= N_VAL_TARGET:
        break
assert val_indices_set and (set(range(len(train_dataset))) - val_indices_set).isdisjoint(val_indices_set)

def make_finetune_record(sample):
    text = normalize_spanish_text(sample.get("sentence", ""))
    audio_path = sample["audio"].get("path")
    if len(text) < 2 or not audio_path:
        return None
    return {
        "text": f"Speaker 1: {text}",
        "original_text": sample.get("sentence", ""),
        "audio": audio_path,
        "client_id": sample.get("client_id", ""),
    }

print("Formateando train/dev speaker-disjoint a JSONL...")
train_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"

val_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
train_written = val_written = rejected = 0
with open(train_jsonl_path, "w", encoding="utf-8") as train_file, \
     open(val_jsonl_path, "w", encoding="utf-8") as val_file:
    for i in tqdm(range(len(train_dataset)), desc="Escribiendo train JSONL"):
        sample = train_dataset[i]
        record = make_finetune_record(sample)
        if record is None:
            rejected += 1
            continue
        if i in val_indices_set:
            val_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            val_written += 1
        else:
            train_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            train_written += 1

split_metadata = {
    "seed": SPLIT_SEED, "train_records": train_written,
    "validation_records": val_written, "validation_speakers": len(val_speakers),
    "rejected_records": rejected, "native_validation_reserved_for": "PTQ calibration",
}
with open(FINETUNE_DATA_DIR / "split_metadata.json", "w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=2, ensure_ascii=False)
print(f"Train: {train_written:,} | Dev: {val_written:,} | Speakers dev: {len(val_speakers):,}")
print(f"Rechazados: {rejected:,} | Metadata: {FINETUNE_DATA_DIR / 'split_metadata.json'}")

---
## TSK-1.5.2: Script de Fine-Tuning para 8GB VRAM

Parametros optimizados para RTX 4060 Ti (8GB):

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `per_device_train_batch_size` | 1 | Limitar VRAM de activaciones |
| `gradient_accumulation_steps` | 64 | Batch efectivo = 64 |
| `bf16` | True | Mixed precision (Ada Lovelace) |
| `ddpm_batch_mul` | 1 | Minimo; sin amplificacion de difusion |
| `lora_r` | 8 | Rank estandar para adaptacion |
| `learning_rate` | 1.0e-5 | Conservador para segundo LoRA |
| `num_train_epochs` | 1 | ~5,263 pasos (~7-15h) |
| `voice_prompt_drop_rate` | 1.0 | Evita prompt extraido del target |

El script generado se guarda en `scripts/run_finetune_es.sh`.

In [ ]:
print("[S1.5-02] Validacion del script de entrenamiento")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = "/home/alfrog/micromamba/envs/vibevoice/bin/python"
MODEL_PATH = str(WEIGHTS_DIR / "vibevoice-1.5b-es")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "finetune_vibevoice_es_correction")
TRAIN_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_train.jsonl")
VAL_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_val.jsonl")

train_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Correccion linguistica VibeVoice-ES (segundo LoRA)
# Hardware: RTX 4060 Ti 8GB VRAM
# Diffusion head y connectors congelados; objetivo CE-only
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Iniciando Fine-Tuning VibeVoice-ES ==="
echo "Modelo base : {MODEL_PATH}"
echo "Train JSONL : {TRAIN_JSONL}"
echo "Val JSONL   : {VAL_JSONL}"
echo "Output      : {OUTPUT_DIR}"
echo ""

# Limpiar cache CUDA antes de empezar
{VENV_PYTHON} -c "import torch; torch.cuda.empty_cache(); print(f'VRAM libre: {{torch.cuda.memory_allocated()/1024**3:.2f}} GB')"

{VENV_PYTHON} -m vibevoice.finetune.train_vibevoice \\
    --model_name_or_path {MODEL_PATH} \\
    --train_jsonl {TRAIN_JSONL} \\
    --validation_jsonl {VAL_JSONL} \\
    --output_dir {OUTPUT_DIR} \\
    --text_column_name text \\
    --audio_column_name audio \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps 64 \\
    --learning_rate 2.5e-5 \\
    --lr_scheduler_type cosine \\
    --warmup_ratio 0.03 \\
    --num_train_epochs 1 \\
    --logging_steps 50 \\
    --save_steps 500 \\
    --eval_steps 500 \\
    --save_total_limit 3 \\
    --bf16 True \\
    --do_train \\
    --do_eval \\
    --dataloader_num_workers 2 \\
    --remove_unused_columns False \\
    --gradient_checkpointing False \\
    --ddpm_batch_mul 1 \\
    --diffusion_loss_weight 1.0 \\
    --train_diffusion_head True \\
    --ce_loss_weight 1.0  # mas peso a precision textual \\
    --voice_prompt_drop_rate 1.0 \\
    --lora_r 8 \\
    --lora_alpha 32 \\
    --lora_dropout 0.05 \\
    --lora_target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj \\
    --max_grad_norm 0.8 \\
    --gradient_clipping \\
    --seed 42

echo ""
echo "=== Fine-Tuning completado ==="
echo "Checkpoint en: {OUTPUT_DIR}"
"""

script_path = SCRIPTS_DIR / "run_finetune_es.sh"
train_script = train_script.replace("--learning_rate 2.5e-5", "--learning_rate 1.0e-5")
train_script = train_script.replace("--logging_steps 50", "--logging_steps 50 \\n    --eval_strategy steps \\n    --save_strategy steps")
train_script = train_script.replace("--diffusion_loss_weight 1.0", "--diffusion_loss_weight 0.0")
train_script = train_script.replace("--train_diffusion_head True", "--train_diffusion_head False \\n    --train_connectors False")
train_script = train_script.replace("--ce_loss_weight 1.0  # mas peso a precision textual", "--ce_loss_weight 1.0")
train_script = train_script.replace("--voice_prompt_drop_rate 1.0", "--voice_prompt_drop_rate 1.0 \\n    --target_audio_dbfs -25.0 \\n    --augment_target_silence False")
# El script versionado es la fuente autoritativa; esta celda no lo sobrescribe
# con una plantilla historica.
if script_path.exists():
    train_script = script_path.read_text(encoding="utf-8")
script_path.write_text(train_script)
script_path.chmod(0o755)
import subprocess
subprocess.run(["bash", "-n", str(script_path)], check=True)

print(f"Script de entrenamiento generado: {script_path}")
print(f"\nPara ejecutar:\n  bash {script_path}")

---
## TSK-1.5.3: Merge de LoRA al Modelo Base

Al finalizar, solo el segundo adaptador LoRA se fusiona sobre VibeVoice-ES.
El diffusion head congelado no se guarda ni se mergea. El resultado BF16 se
escribe en `weights/vibevoice-1.5b-es-corrected/`.

In [ ]:
print("[S1.5-04] Validacion del script de merge")
MERGE_OUTPUT = str(WEIGHTS_DIR / "vibevoice-1.5b-es-corrected")

merge_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Merge LoRA -> VibeVoice-ES
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Mergeando LoRA + Diffusion Head al modelo base ==="
echo "Checkpoint LoRA : {OUTPUT_DIR}"
echo "Modelo base     : {MODEL_PATH}"
echo "Output (ES)     : {MERGE_OUTPUT}"
echo ""

{VENV_PYTHON} -m vibevoice.scripts.merge_vibevoice_models \\
    --base_model_path {MODEL_PATH} \\
    --checkpoint_path {OUTPUT_DIR}/lora \\
    --output_path {MERGE_OUTPUT} \\
    --output_format safetensors

echo ""
echo "=== VibeVoice-ES generado en: {MERGE_OUTPUT} ==="
ls -lh {MERGE_OUTPUT}/*.safetensors 2>/dev/null || echo "(verificar estructura)"
"""

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
merge_script = merge_script.replace("--output_format safetensors", "--output_format safetensors \\n    --output_dtype bfloat16")
if merge_script_path.exists():
    merge_script = merge_script_path.read_text(encoding="utf-8")
merge_script_path.write_text(merge_script)
merge_script_path.chmod(0o755)
subprocess.run(["bash", "-n", str(merge_script_path)], check=True)

print(f"Script de merge generado: {merge_script_path}")
print(f"\nPara ejecutar (despues del entrenamiento):\n  bash {merge_script_path}")

---
## TSK-1.5.4: Validacion del Modelo Fine-Tuneado

Verificamos que **VibeVoice-ES-Corrected** carga en BF16 y registramos VRAM y
tamano. La cuantizacion no debe regenerarse hasta superar el gate WER/CER.

In [ ]:
print("[S1.5-06] Carga del checkpoint VibeVoice-ES-Corrected")
import sys, os
repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es-corrected"

print("Cargando VibeVoice-ES-Corrected...")
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)

vram_es = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES-Corrected cargado. VRAM: {vram_es:.2f} GB")

# --- Verificacion de tamano en disco ---
import pathlib
model_size_mb = sum(f.stat().st_size for f in pathlib.Path(MODEL_ES_PATH).rglob("*.safetensors")) / (1024**2)
print(f"Tamano BF16 en disco: {model_size_mb:.1f} MB")

if vram_es < 6.0:
    print("VRAM OK - margen suficiente para cuantizacion.")
else:
    print("ADVERTENCIA: VRAM elevada. Verificar que el merge fue exitoso.")

In [ ]:
print("[S1.5-07] Gate linguistico espanol antes de cuantizar")
# === Gate linguistico antes de regenerar modelos cuantizados ===
import gc, json, re, time, unicodedata
from pathlib import Path

import jiwer
import librosa
import numpy as np
import soundfile as sf
import whisper

from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

del model_es
gc.collect()
torch.cuda.empty_cache()
quality_model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ES_PATH, torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"}, attn_implementation="sdpa",
)
quality_model.eval()
quality_model.set_ddpm_inference_steps(num_steps=20)
quality_processor = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
quality_texts = [
    "El pingüino camina rápidamente sobre el hielo.",
    "¿Cuándo llegará el próximo tren a la estación?",
    "La educación pública requiere atención y planificación.",
    "Mi hermana compró veintitrés manzanas para la reunión.",
    "¡Qué alegría escuchar música española esta mañana!",
    "Los científicos analizaron cuidadosamente los resultados.",
]
quality_dir = OUTPUTS_DIR / "sprint1_5_quality_gate"
quality_dir.mkdir(parents=True, exist_ok=True)
quality_audios, quality_rtfs = [], []
peak_quality_vram = torch.cuda.memory_allocated() / 1024**3
for index, text in enumerate(quality_texts, start=1):
    inputs = quality_processor(
        text=[f"Speaker 1: {text}"], voice_samples=None, padding=True,
        return_tensors="pt", return_attention_mask=True,
    )
    inputs = {k: (v.to(quality_model.device) if torch.is_tensor(v) else v)
              for k, v in inputs.items() if v is not None}
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.inference_mode():
        output = quality_model.generate(
            **inputs, max_new_tokens=600, tokenizer=quality_processor.tokenizer,
            generation_config={"do_sample": False}, verbose=False,
        )
    elapsed = time.perf_counter() - start
    peak_quality_vram = max(peak_quality_vram, torch.cuda.max_memory_allocated() / 1024**3)
    audio = output.speech_outputs[0].squeeze().cpu().float().numpy()
    duration = len(audio) / 24000
    quality_rtfs.append(elapsed / duration)
    quality_audios.append(audio)
    sf.write(quality_dir / f"sample_{index:02d}.wav", audio, 24000, subtype="PCM_16")

del quality_model, output, inputs
gc.collect()
torch.cuda.empty_cache()
whisper_cache = Path.home() / ".cache" / "whisper"
wm = whisper.load_model("large-v3", device="cuda", download_root=str(whisper_cache))
def normalize_asr_text(text):
    text = unicodedata.normalize("NFC", text.lower())
    text = re.sub(r"[^a-záéíóúüñ0-9 ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

quality_wers, quality_cers, transcripts = [], [], []
for text, audio in zip(quality_texts, quality_audios):
    audio_16k = librosa.resample(audio.astype(np.float32), orig_sr=24000, target_sr=16000)
    transcript = wm.transcribe(
        audio_16k, language="es", fp16=True, verbose=False,
        condition_on_previous_text=False,
    )["text"].strip()
    transcripts.append(transcript)
    reference_norm = normalize_asr_text(text)
    transcript_norm = normalize_asr_text(transcript)
    quality_wers.append(jiwer.wer(reference_norm, transcript_norm))
    quality_cers.append(jiwer.cer(reference_norm, transcript_norm))
    print(f"WER={quality_wers[-1]:.3f} | {transcript}")

del wm
gc.collect()
torch.cuda.empty_cache()
quality_metrics = {
    "wer": float(np.mean(quality_wers)),
    "cer": float(np.mean(quality_cers)),
    "rtf": float(np.mean(quality_rtfs)),
    "vram_peak_gb": peak_quality_vram,
    "samples": [
        {"reference": ref, "transcript": hyp, "wer": wer}
        for ref, hyp, wer in zip(quality_texts, transcripts, quality_wers)
    ],
}
with open(quality_dir / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(quality_metrics, file, indent=2, ensure_ascii=False)
print(f"WER medio={quality_metrics['wer']:.4f} | CER={quality_metrics['cer']:.4f}")
assert quality_metrics["wer"] <= 0.30, (
    "Gate WER fallido: no regenerar cuantizaciones hasta corregir el fine-tuning."
)


## pre-sprint antes de los demas (2,3,4,...)

In [1]:
# ================================================================
# FASE A: Extraccion del Qwen2 (checkpoint standalone)
# Requisito legacy exclusivo de Sprint 4 (AWQ)
# NO ejecutar para Sprints 2-3
# ================================================================

import os, sys, gc, shutil
import torch, torch.nn as nn
from pathlib import Path

for _v in ['model','qwen2_causal']:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path: sys.path.insert(0, repo_path)

import transformers
if hasattr(transformers,"CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer",None)
    if hasattr(transformers.CONFIG_MAPPING,"_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer",None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
from transformers import Qwen2ForCausalLM

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es-corrected"
QWEN2_STANDALONE = Path("../weights/standalone_qwen_fp16")

print("Cargando VibeVoice-ES...")
model = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH, torch_dtype=torch.float16,
    device_map={"": "cuda:0"}, attn_implementation="sdpa",
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Extraer Qwen2Model y crear Qwen2ForCausalLM
qwen2_causal = Qwen2ForCausalLM(model.model.language_model.config)
qwen2_causal.model.load_state_dict(model.model.language_model.state_dict())
# lm_head tied con embed_tokens
qwen2_causal.lm_head.weight = model.model.language_model.embed_tokens.weight
print("Qwen2ForCausalLM ensamblado")

# Guardar como checkpoint standalone
if QWEN2_STANDALONE.exists(): shutil.rmtree(QWEN2_STANDALONE)
QWEN2_STANDALONE.mkdir(parents=True, exist_ok=True)
qwen2_causal.save_pretrained(str(QWEN2_STANDALONE), safe_serialization=True)

# Guardar tokenizer (necesario para calibracion de texto)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
processor.tokenizer.save_pretrained(str(QWEN2_STANDALONE))

del model, qwen2_causal, processor
gc.collect(); torch.cuda.empty_cache()
print(f"Qwen2 standalone guardado en: {QWEN2_STANDALONE}")
for f in sorted(QWEN2_STANDALONE.glob("*.safetensors")):
    print(f"  {f.name}: {f.stat().st_size/(1024**2):.0f} MB")


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
APEX FusedRMSNorm not available, using native implementation
/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando VibeVoice-ES...
Tied input and output embeddings using standard assignment.


Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.01s/it]


Tied input and output embeddings using standard assignment.
VRAM: 5.04 GB
Qwen2ForCausalLM ensamblado


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Qwen2 standalone guardado en: ../weights/standalone_qwen_fp16
  model-00001-of-00002.safetensors: 4765 MB
  model-00002-of-00002.safetensors: 1569 MB


---
# Sprint 2: GGUF IQ4_NL selectivo con runtime C++

> Implementacion reproducible sobre CrispASR `v0.8.23`, no sobre `ggc v6`.
> Solo las 196 matrices de atencion/MLP de Qwen se almacenan como IQ4_NL.
> Tokenizadores, conectores, diffusion head y embeddings permanecen en F16/F32.
> Salida: `weights/vibevoice-1.5b-es-iq4_nl/vibevoice-1.5b-iq4_nl.gguf`.

El artefacto publico de gguf-org mide 3.536 GB y `ggc v6` vuelve a PyTorch; no respalda la afirmacion de menos de 1.2 GB. Este sprint usa el runtime GGML/C++ real de CrispASR y valida el tipo de cada tensor antes de aceptar el archivo.


In [ ]:
print("[S2-01] Preflight GGUF IQ4_NL selectivo")
import json
from pathlib import Path

MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
IQ4_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-iq4_nl"
IQ4_MODEL = IQ4_OUTPUT / "vibevoice-1.5b-iq4_nl.gguf"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint corregido. Complete S1.5-03, S1.5-05 y S1.5-07."
)
with (MODEL_ES_PATH / "config.json").open(encoding="utf-8") as f:
    cfg = json.load(f)
decoder = cfg["decoder_config"]
assert decoder["hidden_size"] == 1536
assert decoder["num_hidden_layers"] == 28
assert decoder["num_attention_heads"] == 12
print("Arquitectura compatible: Qwen2 1.5B, 28 capas, 196 matrices objetivo")
print("Cota inferior del artefacto selectivo: ~3.28 GiB; <1.2 GB no es posible con audio F16")


In [ ]:
print("[S2-02] Conversion y cuantizacion IQ4_NL")
import subprocess
subprocess.run(
    ["bash", str(SCRIPTS_DIR / "build_vibevoice_iq4_nl.sh"), str(MODEL_ES_PATH), str(IQ4_OUTPUT)],
    cwd=PROJECT_ROOT, check=True,
)
assert IQ4_MODEL.is_file()
print(f"Modelo guardado: {IQ4_MODEL} ({IQ4_MODEL.stat().st_size / 1e9:.3f} GB)")


In [ ]:
print("[S2-03] Smoke test del runtime GGML/C++")
import subprocess
IQ4_SMOKE_WAV = OUTPUTS_DIR / "sprint2_iq4_nl" / "smoke-es.wav"
subprocess.run(
    ["bash", str(SCRIPTS_DIR / "smoke_vibevoice_iq4_nl.sh"), str(IQ4_MODEL), str(IQ4_SMOKE_WAV)],
    cwd=PROJECT_ROOT, check=True,
)
assert IQ4_SMOKE_WAV.is_file() and IQ4_SMOKE_WAV.stat().st_size > 44
print(f"Audio nativo guardado: {IQ4_SMOKE_WAV}")


---
# Sprint 3: GPTQ W4 g128 selectivo y recargable

> GPTQModel 2.2.0 + Transformers 4.51.3 en el entorno `vibevoice`.
> Solo las 196 proyecciones Qwen usan GPTQ W4; audio, connectors y diffusion head quedan BF16.
> El artefacto se acepta solo tras recargar los `QuantLinear` empaquetados en un proceso limpio.
> AutoGPTQ no se usa porque esta archivado y no mantiene Python 3.12/CUDA 13.


In [ ]:
print("[S3-01] Preflight GPTQ selectivo")
import json
from pathlib import Path

import gptqmodel
import transformers

MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
GPTQ_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-gptq"
GPTQ_METADATA = DATA_DIR / "calibration_metadata.json"
assert gptqmodel.__version__ == "2.2.0", (
    "Ejecute bash scripts/setup_gptqmodel.sh"
)
assert transformers.__version__ == "4.51.3"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint corregido aprobado por S1.5-07"
)
assert GPTQ_METADATA.is_file()
with GPTQ_METADATA.open(encoding="utf-8") as file:
    calibration_info = json.load(file)
assert calibration_info["split_origin"] == "validation"
print("GPTQModel 2.2.0 + Triton; objetivo: 196 proyecciones Qwen")


In [ ]:
print("[S3-02] Extraccion BF16, calibracion y cuantizacion GPTQ")
import subprocess
subprocess.run(
    [
        "bash", str(SCRIPTS_DIR / "build_vibevoice_gptq.sh"),
        str(MODEL_ES_PATH), str(GPTQ_OUTPUT), str(GPTQ_METADATA),
    ],
    cwd=PROJECT_ROOT, check=True,
)
with (GPTQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    gptq_manifest = json.load(file)
assert gptq_manifest["status"] == "structure_validated"
assert gptq_manifest["validation"]["gptq_modules"] == 196
print(f"Modelo GPTQ guardado: {GPTQ_OUTPUT}")


In [ ]:
print("[S3-03] Recarga limpia y smoke TTS del artefacto GPTQ")
import subprocess
GPTQ_SMOKE_WAV = OUTPUTS_DIR / "sprint3_gptq" / "smoke-es.wav"
subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
        "--model", str(GPTQ_OUTPUT), "--output", str(GPTQ_SMOKE_WAV),
    ],
    cwd=PROJECT_ROOT, check=True,
)
assert GPTQ_SMOKE_WAV.is_file() and GPTQ_SMOKE_WAV.stat().st_size > 44
with GPTQ_SMOKE_WAV.with_suffix(".json").open(encoding="utf-8") as file:
    gptq_smoke = json.load(file)
assert gptq_smoke["gptq_modules"] == 196
with (GPTQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    assert json.load(file)["status"] == "validated"
print(json.dumps(gptq_smoke, indent=2))


---
# Sprint 4: AWQ via AutoAWQ (ncoder-ai, todo en vibevoice)

> ncoder-ai: AutoAWQ zero_point=True, g128, w4, GEMM.
> AutoAWQ instalado en vibevoice (wheel pre-compilado). Sin qforge.
> Paso A: AutoAWQ.quantize() Qwen2  |  Paso B: inyeccion + benchmark.


In [3]:
# === Sprint 4: AWQ via AutoAWQ (ncoder-ai, todo en vibevoice) ===

# ncoder-ai: AutoAWQ con zero_point=True, g128, w4, GEMM, modules_to_not_convert
# AutoAWQ instalado en vibevoice (wheel pre-compilado, sin qforge)

import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path

for _v in ["model","model_awq","qwen2","decoder_awq"]:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

QWEN2_STANDALONE = Path("../weights/standalone_qwen_fp16")
AWQ_OUTPUT = Path("../weights/vibevoice-1.5b-es-awq")
MODEL_ES_PATH = "../weights/vibevoice-1.5b-es-corrected"

# Paso A: Cuantizar Qwen2 standalone con AutoAWQ
_awq_ckpt = AWQ_OUTPUT / "decoder-awq"
if _awq_ckpt.exists() and any(_awq_ckpt.iterdir()):
    print(f"Paso A ya completado: {_awq_ckpt}")
else:
    print("=== Paso A: AutoAWQ quantize Qwen2 ===\n")
    from datasets import load_dataset, Dataset
    from transformers import AutoTokenizer
    from awq import AutoAWQForCausalLM
    import glob as _glob

    cv17 = load_dataset("fsicoli/common_voice_17_0","es",split="train")
    cache_dir = os.path.expanduser("~/.cache/huggingface/datasets/fsicoli___common_voice_17_0/es/16.1.0")
    arrow_files = _glob.glob(cache_dir + "/**/*.arrow", recursive=True)
    ds = Dataset.from_file(sorted(arrow_files)[0])
    sentences = [s for s in ds["sentence"] if isinstance(s,str) and len(s)>20][:3000]
    chunk, chunk_len, calib = [], 0, []
    for s in sentences:
        chunk.append(s); chunk_len += len(s.split())
        if chunk_len >= 200: calib.append(" ".join(chunk)); chunk, chunk_len = [], 0
    if chunk: calib.append(" ".join(chunk))
    print(f"Calib: {len(calib)} chunks")

    model = AutoAWQForCausalLM.from_pretrained(str(QWEN2_STANDALONE), safetensors=True)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)  # base tokenizer
    quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
    model.quantize(tokenizer, quant_config=quant_config)

    (AWQ_OUTPUT / "decoder-awq").mkdir(parents=True, exist_ok=True)
    model.save_quantized(str(AWQ_OUTPUT / "decoder-awq"))
    tokenizer.save_pretrained(str(AWQ_OUTPUT / "decoder-awq"))
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"AWQ done: {AWQ_OUTPUT / 'decoder-awq'}")

# Paso B: Inyeccion + benchmark (mismo patron que Sprint 3)
print("\n=== Paso B: Cargar decoder AWQ + VibeVoice + benchmark ===\n")
sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
from awq import AutoAWQForCausalLM
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper
print("Cargando decoder AWQ con AutoAWQ nativo...")
decoder_awq = AutoAWQForCausalLM.from_quantized(
    str(AWQ_OUTPUT / "decoder-awq"),
    device_map={"": "cuda:0"},
)
vram_dec = torch.cuda.memory_allocated() / 1024**3
print(f"Decoder AWQ (INT4): {vram_dec:.2f} GB")

print("Cargando VibeVoice-ES...")
model_inf = VibeVoiceForConditionalGenerationInference.from_pretrained(MODEL_ES_PATH, torch_dtype=torch.float16, device_map={"": "cuda:0"}, attn_implementation="sdpa")
model_inf.eval(); model_inf.set_ddpm_inference_steps(num_steps=20)
# Inyeccion: language_model = decoder AWQ (WQLinear layers)
model_inf.model.language_model = decoder_awq.model.model
model_inf.lm_head.weight = decoder_awq.model.model.embed_tokens.weight
del decoder_awq; gc.collect(); torch.cuda.empty_cache()
vram_idle = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM reposo AWQ: {vram_idle:.2f} GB")

# Guardar modelo portable
model_inf.save_pretrained(str(AWQ_OUTPUT), safe_serialization=True)
proc_tmp = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
proc_tmp.save_pretrained(str(AWQ_OUTPUT))
del proc_tmp
print(f"Modelo portable guardado: {AWQ_OUTPUT}")

# Benchmark TTS
TARGET_SR=24000
test_texts=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
processor_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
all_wer=[]; all_cer=[]; all_rtf=[]; wm=whisper.load_model("large-v3")
for test_text in test_texts:
    t_start=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    inp=processor_inf(text=[f"Speaker 1: {test_text}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
    inp={k:(v.to(model_inf.device) if torch.is_tensor(v) else v) for k,v in inp.items() if v is not None}
    with torch.no_grad(): out=model_inf.generate(**inp,max_new_tokens=600,tokenizer=processor_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
    t_end=time.perf_counter()
    if out.speech_outputs and out.speech_outputs[0] is not None:
        a=out.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR; all_rtf.append((t_end-t_start)/d)
        a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
        trans=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
        all_wer.append(jiwer.wer(test_text.lower(),trans.lower())); all_cer.append(jiwer.cer(test_text.lower(),trans.lower()))
        print(f"  Audio: {d:.1f}s | RTF: {all_rtf[-1]:.2f} | WER={all_wer[-1]:.3f} | {trans[:60]}")
    else:
        all_wer.append(1.0); all_cer.append(1.0); all_rtf.append(float("inf")); print("  ERROR")
vram_peak=torch.cuda.max_memory_allocated()/1024**3; rtf=np.mean(all_rtf) if all_rtf else float("inf")
wer_awq=np.mean(all_wer) if all_wer else 1.0; cer_awq=np.mean(all_cer) if all_cer else 1.0
del wm,out,model_inf; gc.collect(); torch.cuda.empty_cache()
print(f"\n=== AWQ === VRAM peak: {vram_peak:.2f} GB | RTF: {rtf:.4f} | WER: {wer_awq:.4f} | CER: {cer_awq:.4f}")



Paso A ya completado: ../weights/vibevoice-1.5b-es-awq/decoder-awq

=== Paso B: Cargar decoder AWQ + VibeVoice + benchmark ===



/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando decoder AWQ con AutoAWQ nativo...


Replacing layers...: 100%|██████████| 28/28 [00:03<00:00,  7.69it/s]
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)


Decoder AWQ (INT4): 1.09 GB
Cargando VibeVoice-ES...


...

  Audio: 7.9s | RTF: 1.90 | WER=1.154 | Hola, buenos días. Es el pueblo muy unido entre los formador


Generating (active: 1/1):   0%|          | 0/92 [00:00<?, ?it/s]/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/schedule/dpm_solver.py:384: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sigmas = np.array(((1 - self.alphas_cumprod) / self.alphas_cumprod) ** 0.5)


  Audio: 6.8s | RTF: 1.40 | WER=0.333 | La inteligencia artificial también crea sistemas de box cada


  Audio: 7.6s | RTF: 1.45 | WER=1.000 | Ahora el mensaje profundo ha reconocido la lengua de la dict


  Audio: 7.9s | RTF: 1.33 | WER=0.167 | Los modelos de lenguaje pueden generar texto y voz a gran pr

=== AWQ === VRAM peak: 9.48 GB | RTF: 1.5228 | WER: 0.6635 | CER: 0.3978


---
# Sprint 5: INT8 Hibrido Selectivo via bitsandbytes

> Replica el enfoque de Fabio Sarracino sobre VibeVoice 1.5B: cuantiza unicamente
> las capas `nn.Linear` del decoder Qwen2 con `BitsAndBytesConfig(load_in_8bit=True)`.
> El cabezal de difusion (`prediction_head` en este fork), los tokenizers, los
> connectors y `lm_head` permanecen en FP16 para proteger los latentes acusticos.

No se usa split layout: convertir el `state_dict` INT8 a FP16 destruiria la
cuantizacion. `save_pretrained()` conserva directamente los pesos y metadatos bnb.


In [ ]:
# === Sprint 5: INT8 hibrido selectivo (Fabio Sarracino) ===
import os, sys, gc, json, shutil, time
from pathlib import Path

import librosa
import jiwer
import numpy as np
import soundfile as sf
import torch
import whisper

for _v in ["model", "model_int8", "m_int8", "wm", "out"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Parche obligatorio antes de importar las clases VibeVoice.
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from bitsandbytes.nn import Linear8bitLt
from transformers import BitsAndBytesConfig
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ES_PATH = Path("../weights/vibevoice-1.5b-es-corrected")
INT8_OUT = Path("../weights/vibevoice-1.5b-es-int8")
SPRINT5_OUTPUT = Path("../outputs/sprint5_int8")
SPRINT5_OUTPUT.mkdir(parents=True, exist_ok=True)
TARGET_SR = 24000

# En el fork actual diffusion_head se instancia como model.prediction_head.
SENSITIVE_MODULES = [
    "prediction_head",
    "acoustic_tokenizer",
    "semantic_tokenizer",
    "acoustic_connector",
    "semantic_connector",
    "lm_head",
]
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_skip_modules=SENSITIVE_MODULES,
)

print("Cargando VibeVoice-ES con INT8 selectivo sobre Qwen2...")
model_int8 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    str(MODEL_ES_PATH),
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
model_int8.eval()
model_int8.set_ddpm_inference_steps(num_steps=20)

# Verificacion estructural: solo el language_model puede contener Linear8bitLt.
quantized_linears = [
    name for name, module in model_int8.named_modules()
    if isinstance(module, Linear8bitLt)
]
sensitive_quantized = [
    name for name in quantized_linears
    if any(part in name for part in SENSITIVE_MODULES)
]
unexpected_quantized = [
    name for name in quantized_linears
    if not name.startswith("model.language_model.")
]
assert quantized_linears, "No se encontro ninguna capa Linear8bitLt; revisar bitsandbytes/CUDA."
assert not sensitive_quantized, f"Modulos sensibles cuantizados: {sensitive_quantized[:5]}"
assert not unexpected_quantized, f"Capas fuera de Qwen2 cuantizadas: {unexpected_quantized[:5]}"
assert model_int8.lm_head.weight is model_int8.model.language_model.embed_tokens.weight, (
    "lm_head y embed_tokens dejaron de compartir pesos."
)

quantized_weight_ids = {
    id(module.weight) for module in model_int8.modules()
    if isinstance(module, Linear8bitLt)
}
total_params = sum(param.numel() for param in model_int8.parameters())
int8_params = sum(
    param.numel() for param in model_int8.parameters()
    if id(param) in quantized_weight_ids
)
int8_fraction = int8_params / total_params
vram_int8_idle = torch.cuda.memory_allocated() / 1024**3
print(f"Capas Linear8bitLt       : {len(quantized_linears)}")
print(f"Parametros INT8          : {int8_params:,} ({int8_fraction:.1%})")
print(f"VRAM reposo INT8 hibrido: {vram_int8_idle:.2f} GB")
print("Audio modules         : FP16 (verificacion OK)")

# Guardar el modelo bnb directamente. No convertir el state_dict a FP16.
if INT8_OUT.exists():
    shutil.rmtree(INT8_OUT)
model_int8.save_pretrained(str(INT8_OUT), safe_serialization=True)
processor = VibeVoiceProcessor.from_pretrained(str(MODEL_ES_PATH))
processor.save_pretrained(str(INT8_OUT))
disk_gb = sum(path.stat().st_size for path in INT8_OUT.rglob("*.safetensors")) / 1024**3
print(f"Modelo INT8 guardado    : {INT8_OUT} ({disk_gb:.2f} GB en safetensors)")

# Recargar desde disco para validar que el artefacto conserva la cuantizacion.
del model_int8
gc.collect()
torch.cuda.empty_cache()
m_int8 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    str(INT8_OUT),
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
m_int8.eval()
m_int8.set_ddpm_inference_steps(num_steps=20)
reloaded_quantized_names = [
    name for name, module in m_int8.named_modules()
    if isinstance(module, Linear8bitLt)
]
assert len(reloaded_quantized_names) == len(quantized_linears), (
    f"La recarga cambio las capas INT8: {len(quantized_linears)} -> {len(reloaded_quantized_names)}"
)
vram_int8_idle = torch.cuda.memory_allocated() / 1024**3
print(f"Recarga INT8 verificada : {len(reloaded_quantized_names)} capas, {vram_int8_idle:.2f} GB")

# Smoke benchmark de 4 frases. El benchmark estadistico de 200 muestras es Sprint 8.
test_texts = [
    "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.",
    "La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.",
    "El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.",
    "Los modelos de lenguaje pueden generar texto y voz con gran precision.",
]
audios = []
rtfs = []
peak_vram = vram_int8_idle
for text in test_texts:
    inputs = processor(
        text=[f"Speaker 1: {text}"],
        voice_samples=None,
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )
    inputs = {
        key: (value.to(m_int8.device) if torch.is_tensor(value) else value)
        for key, value in inputs.items() if value is not None
    }
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.inference_mode():
        out = m_int8.generate(
            **inputs,
            max_new_tokens=600,
            tokenizer=processor.tokenizer,
            generation_config={"do_sample": False},
            verbose=False,
        )
    elapsed = time.perf_counter() - start
    peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / 1024**3)
    if out.speech_outputs and out.speech_outputs[0] is not None:
        audio = out.speech_outputs[0].squeeze().cpu().float().numpy()
        duration = len(audio) / TARGET_SR
        rtfs.append(elapsed / duration)
        audios.append(audio)
        audio_path = SPRINT5_OUTPUT / f"sample_{len(audios):02d}.wav"
        sf.write(audio_path, audio, TARGET_SR, subtype="PCM_16")
        print(f"Audio: {duration:.1f}s | RTF: {rtfs[-1]:.2f}")
    else:
        audios.append(None)
        print("ERROR: generacion sin audio")

# Liberar todas las referencias CUDA antes de cargar Whisper. No conservar listas
# de modulos: cada objeto Linear8bitLt mantiene vivo su peso INT8.
del m_int8, out, inputs, reloaded_quantized_names
gc.collect()
torch.cuda.empty_cache()
vram_before_asr = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM tras liberar VibeVoice: {vram_before_asr:.2f} GB")
if vram_before_asr > 0.5:
    print("ADVERTENCIA: quedan referencias CUDA activas antes de Whisper.")

whisper_cache = Path.home() / ".cache" / "whisper"
whisper_checkpoint = whisper_cache / "large-v3.pt"
if not whisper_checkpoint.exists():
    raise FileNotFoundError(
        f"Falta {whisper_checkpoint}. Descargue large-v3 por separado para evitar "
        "que el benchmark quede esperando una descarga sin progreso visible."
    )
print(f"Cargando Whisper large-v3 desde cache ({whisper_checkpoint.stat().st_size / 1024**3:.2f} GB)...")
asr_load_start = time.perf_counter()
wm = whisper.load_model("large-v3", device="cuda", download_root=str(whisper_cache))
print(f"Whisper cargado en {time.perf_counter() - asr_load_start:.1f}s")
wers = []
cers = []
for index, (text, audio) in enumerate(zip(test_texts, audios), start=1):
    if audio is None:
        wers.append(1.0)
        cers.append(1.0)
        continue
    print(f"Transcribiendo muestra {index}/{len(audios)}...", flush=True)
    audio_16k = librosa.resample(audio.astype(np.float32), orig_sr=TARGET_SR, target_sr=16000)
    transcript = wm.transcribe(
        audio_16k, language="es", fp16=True, verbose=False,
        condition_on_previous_text=False,
    )["text"].strip()
    wers.append(jiwer.wer(text.lower(), transcript.lower()))
    cers.append(jiwer.cer(text.lower(), transcript.lower()))
    print(f"[{index}] WER={wers[-1]:.3f} CER={cers[-1]:.3f} | {transcript[:80]}")

del wm
gc.collect()
torch.cuda.empty_cache()
rtf_int8 = float(np.mean(rtfs)) if rtfs else float("inf")
wer_int8 = float(np.mean(wers)) if wers else 1.0
cer_int8 = float(np.mean(cers)) if cers else 1.0
metrics_int8 = {
    "model": "vibevoice-1.5b-es-int8-selective",
    "int8_layers": len(quantized_linears),
    "int8_parameter_fraction": int8_fraction,
    "disk_gb": disk_gb,
    "vram_idle_gb": vram_int8_idle,
    "vram_peak_gb": peak_vram,
    "rtf": rtf_int8,
    "wer": wer_int8,
    "cer": cer_int8,
}
with open(SPRINT5_OUTPUT / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_int8, file, indent=2)
print(
    f"\n=== INT8 HIBRIDO === VRAM idle: {vram_int8_idle:.2f} GB | "
    f"VRAM peak: {peak_vram:.2f} GB | RTF: {rtf_int8:.4f} | "
    f"WER: {wer_int8:.4f} | CER: {cer_int8:.4f}"
)


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
APEX FusedRMSNorm not available, using native implementation
/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando VibeVoice-ES con INT8 selectivo sobre Qwen2...


Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.04s/it]


Capas Linear8bitLt       : 196
Parametros INT8          : 1,310,195,712 (48.5%)
VRAM reposo INT8 hibrido: 3.83 GB
Audio modules         : FP16 (verificacion OK)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Modelo INT8 guardado    : ../weights/vibevoice-1.5b-es-int8 (3.82 GB en safetensors)
Recarga INT8 verificada : 196 capas, 3.84 GB


Audio: 6.4s | RTF: 2.56


Audio: 6.7s | RTF: 2.21


Audio: 7.5s | RTF: 2.24


Audio: 7.7s | RTF: 2.22


---
# Sprint 6: NF4 Hibrido + Double Quant via bitsandbytes

> Cuantiza unicamente las 196 proyecciones lineales del Qwen2 con NF4 y doble
> cuantizacion. `embed_tokens`, `lm_head` y todo el pipeline acustico permanecen
> en BF16; las multiplicaciones NF4 tambien usan BF16 como compute dtype.

`embed_tokens` es `nn.Embedding`, por lo que bitsandbytes no lo reemplazaria de
todos modos; se incluye en la lista para documentar y verificar la proteccion.
No se usa split layout ni se exporta un state dict dequantizado a FP16.


In [ ]:
# === Sprint 6: NF4 hibrido + Double Quant ===
import gc, json, os, shutil, sys, time
from pathlib import Path

import jiwer
import librosa
import numpy as np
import soundfile as sf
import torch
import whisper

for _v in ["model", "model_nf4", "m_nf4", "wm", "out"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Parche obligatorio antes de importar las clases VibeVoice.
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from bitsandbytes.nn import Linear4bit
from transformers import BitsAndBytesConfig
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ES_PATH = Path("../weights/vibevoice-1.5b-es-corrected")
NF4_OUT = Path("../weights/vibevoice-1.5b-es-nf4")
SPRINT6_OUTPUT = Path("../outputs/sprint6_nf4")
SPRINT6_OUTPUT.mkdir(parents=True, exist_ok=True)
TARGET_SR = 24000

# embed_tokens es nn.Embedding (bnb no lo cuantiza), pero se declara para dejar
# explicita la proteccion. lm_head comparte ese mismo peso.
SENSITIVE_MODULES_NF4 = [
    "embed_tokens",
    "lm_head",
    "prediction_head",
    "acoustic_tokenizer",
    "semantic_tokenizer",
    "acoustic_connector",
    "semantic_connector",
]
bnb_nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.uint8,
    llm_int8_skip_modules=SENSITIVE_MODULES_NF4,
)

print("Cargando VibeVoice-ES con NF4 selectivo sobre Qwen2...")
model_nf4 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    str(MODEL_ES_PATH),
    quantization_config=bnb_nf4_config,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
model_nf4.eval()
model_nf4.set_ddpm_inference_steps(num_steps=20)

# Verificar ubicacion, formato, doble cuantizacion y compute dtype.
quantized_linears_nf4 = [
    name for name, module in model_nf4.named_modules()
    if isinstance(module, Linear4bit)
]
sensitive_quantized_nf4 = [
    name for name in quantized_linears_nf4
    if any(part in name for part in SENSITIVE_MODULES_NF4)
]
unexpected_quantized_nf4 = [
    name for name in quantized_linears_nf4
    if not name.startswith("model.language_model.")
]
nf4_modules = [
    module for module in model_nf4.modules()
    if isinstance(module, Linear4bit)
]
assert quantized_linears_nf4, "No se encontro ninguna capa Linear4bit."
assert not sensitive_quantized_nf4, f"Modulos sensibles cuantizados: {sensitive_quantized_nf4[:5]}"
assert not unexpected_quantized_nf4, f"Capas fuera de Qwen2 cuantizadas: {unexpected_quantized_nf4[:5]}"
assert all(module.compute_dtype == torch.bfloat16 for module in nf4_modules), (
    "Hay capas NF4 cuyo compute dtype no es BF16."
)
assert all(module.weight.quant_state is not None for module in nf4_modules), (
    "Hay capas Linear4bit sin estado de cuantizacion."
)
double_quant_layers = sum(
    bool(getattr(module.weight.quant_state, "nested", False))
    for module in nf4_modules
)
assert double_quant_layers == len(nf4_modules), (
    f"Double Quant solo esta activo en {double_quant_layers}/{len(nf4_modules)} capas."
)
assert isinstance(model_nf4.model.language_model.embed_tokens, torch.nn.Embedding)
assert model_nf4.model.language_model.embed_tokens.weight.dtype == torch.bfloat16
assert isinstance(model_nf4.lm_head, torch.nn.Linear)
assert model_nf4.lm_head.weight is model_nf4.model.language_model.embed_tokens.weight, (
    "lm_head y embed_tokens dejaron de compartir pesos."
)

quantized_weight_ids_nf4 = {id(module.weight) for module in nf4_modules}
total_params_nf4 = sum(param.numel() for param in model_nf4.parameters())
nf4_params = sum(
    param.numel() for param in model_nf4.parameters()
    if id(param) in quantized_weight_ids_nf4
)
nf4_fraction = nf4_params / total_params_nf4
vram_nf4_idle = torch.cuda.memory_allocated() / 1024**3
print(f"Capas Linear4bit NF4    : {len(quantized_linears_nf4)}")
print(f"Capas con Double Quant : {double_quant_layers}")
print(f"Parametros NF4         : {nf4_params:,} ({nf4_fraction:.1%})")
print(f"embed_tokens / lm_head : BF16 tied (verificacion OK)")
print(f"Audio modules          : BF16 (verificacion OK)")
print(f"VRAM reposo NF4 hibrido: {vram_nf4_idle:.2f} GB")

# Guardado nativo: conservar Params4bit y quant_state; nunca castear a FP16.
if NF4_OUT.exists():
    shutil.rmtree(NF4_OUT)
model_nf4.save_pretrained(str(NF4_OUT), safe_serialization=True)
processor_nf4 = VibeVoiceProcessor.from_pretrained(str(MODEL_ES_PATH))
processor_nf4.save_pretrained(str(NF4_OUT))
disk_nf4_gb = sum(path.stat().st_size for path in NF4_OUT.rglob("*.safetensors")) / 1024**3
print(f"Modelo NF4 guardado     : {NF4_OUT} ({disk_nf4_gb:.2f} GB en safetensors)")

# nf4_modules contiene objetos CUDA; eliminarlo antes de liberar el modelo.
del nf4_modules, model_nf4
gc.collect()
torch.cuda.empty_cache()
m_nf4 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    str(NF4_OUT),
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
m_nf4.eval()
m_nf4.set_ddpm_inference_steps(num_steps=20)
reloaded_nf4_names = [
    name for name, module in m_nf4.named_modules()
    if isinstance(module, Linear4bit)
]
reloaded_double_quant = sum(
    bool(getattr(module.weight.quant_state, "nested", False))
    for module in m_nf4.modules() if isinstance(module, Linear4bit)
)
assert len(reloaded_nf4_names) == len(quantized_linears_nf4)
assert reloaded_double_quant == len(reloaded_nf4_names)
assert m_nf4.model.language_model.embed_tokens.weight.dtype == torch.bfloat16
assert m_nf4.lm_head.weight is m_nf4.model.language_model.embed_tokens.weight
vram_nf4_idle = torch.cuda.memory_allocated() / 1024**3
print(
    f"Recarga NF4 verificada  : {len(reloaded_nf4_names)} capas, "
    f"Double Quant={reloaded_double_quant}, {vram_nf4_idle:.2f} GB"
)

# Smoke benchmark de 4 frases. Sprint 8 realizara la evaluacion estadistica.
test_texts_nf4 = [
    "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.",
    "La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.",
    "El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.",
    "Los modelos de lenguaje pueden generar texto y voz con gran precision.",
]
audios_nf4 = []
rtfs_nf4 = []
peak_vram_nf4 = vram_nf4_idle
for text in test_texts_nf4:
    inputs_nf4 = processor_nf4(
        text=[f"Speaker 1: {text}"],
        voice_samples=None,
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )
    inputs_nf4 = {
        key: (value.to(m_nf4.device) if torch.is_tensor(value) else value)
        for key, value in inputs_nf4.items() if value is not None
    }
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.inference_mode():
        out = m_nf4.generate(
            **inputs_nf4,
            max_new_tokens=600,
            tokenizer=processor_nf4.tokenizer,
            generation_config={"do_sample": False},
            verbose=False,
        )
    elapsed = time.perf_counter() - start
    peak_vram_nf4 = max(peak_vram_nf4, torch.cuda.max_memory_allocated() / 1024**3)
    if out.speech_outputs and out.speech_outputs[0] is not None:
        audio = out.speech_outputs[0].squeeze().cpu().float().numpy()
        duration = len(audio) / TARGET_SR
        rtfs_nf4.append(elapsed / duration)
        audios_nf4.append(audio)
        audio_path = SPRINT6_OUTPUT / f"sample_{len(audios_nf4):02d}.wav"
        sf.write(audio_path, audio, TARGET_SR, subtype="PCM_16")
        print(f"Audio: {duration:.1f}s | RTF: {rtfs_nf4[-1]:.2f}")
    else:
        audios_nf4.append(None)
        print("ERROR: generacion sin audio")

# Liberar VibeVoice y cualquier referencia CUDA antes de cargar Whisper large-v3.
del m_nf4, out, inputs_nf4, reloaded_nf4_names
gc.collect()
torch.cuda.empty_cache()
vram_before_asr_nf4 = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM tras liberar VibeVoice: {vram_before_asr_nf4:.2f} GB")
if vram_before_asr_nf4 > 0.5:
    print("ADVERTENCIA: quedan referencias CUDA activas antes de Whisper.")

whisper_cache = Path.home() / ".cache" / "whisper"
whisper_checkpoint = whisper_cache / "large-v3.pt"
if not whisper_checkpoint.exists():
    raise FileNotFoundError(
        f"Falta {whisper_checkpoint}. Descargue large-v3 por separado para evitar "
        "una descarga sin progreso visible durante el benchmark."
    )
print(f"Cargando Whisper large-v3 desde cache ({whisper_checkpoint.stat().st_size / 1024**3:.2f} GB)...")
asr_load_start = time.perf_counter()
wm = whisper.load_model("large-v3", device="cuda", download_root=str(whisper_cache))
print(f"Whisper cargado en {time.perf_counter() - asr_load_start:.1f}s")
wers_nf4 = []
cers_nf4 = []
for index, (text, audio) in enumerate(zip(test_texts_nf4, audios_nf4), start=1):
    if audio is None:
        wers_nf4.append(1.0)
        cers_nf4.append(1.0)
        continue
    print(f"Transcribiendo muestra {index}/{len(audios_nf4)}...", flush=True)
    audio_16k = librosa.resample(audio.astype(np.float32), orig_sr=TARGET_SR, target_sr=16000)
    transcript = wm.transcribe(
        audio_16k, language="es", fp16=True, verbose=False,
        condition_on_previous_text=False,
    )["text"].strip()
    wers_nf4.append(jiwer.wer(text.lower(), transcript.lower()))
    cers_nf4.append(jiwer.cer(text.lower(), transcript.lower()))
    print(f"[{index}] WER={wers_nf4[-1]:.3f} CER={cers_nf4[-1]:.3f} | {transcript[:80]}")

del wm
gc.collect()
torch.cuda.empty_cache()
rtf_nf4 = float(np.mean(rtfs_nf4)) if rtfs_nf4 else float("inf")
wer_nf4 = float(np.mean(wers_nf4)) if wers_nf4 else 1.0
cer_nf4 = float(np.mean(cers_nf4)) if cers_nf4 else 1.0
metrics_nf4 = {
    "model": "vibevoice-1.5b-es-nf4-selective",
    "quant_type": "nf4",
    "double_quant": True,
    "compute_dtype": "bfloat16",
    "nf4_layers": len(quantized_linears_nf4),
    "nf4_parameter_fraction": nf4_fraction,
    "disk_gb": disk_nf4_gb,
    "vram_idle_gb": vram_nf4_idle,
    "vram_peak_gb": peak_vram_nf4,
    "rtf": rtf_nf4,
    "wer": wer_nf4,
    "cer": cer_nf4,
}
with open(SPRINT6_OUTPUT / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_nf4, file, indent=2)
print(
    f"\n=== NF4 HIBRIDO === VRAM idle: {vram_nf4_idle:.2f} GB | "
    f"VRAM peak: {peak_vram_nf4:.2f} GB | RTF: {rtf_nf4:.4f} | "
    f"WER: {wer_nf4:.4f} | CER: {cer_nf4:.4f}"
)


---
# Sprint 7: FP8 E4M3FN con Escalamiento Dinamico

> Las 196 proyecciones lineales de Qwen almacenan pesos FP8 escalados por capa.
> Cada forward escala dinamicamente las activaciones, ejecuta `torch._scaled_mm`
> FP8 nativo y devuelve BF16. Audio, embeddings y `lm_head` permanecen en BF16.

PyTorch reporta un maximo finito de 448 para `float8_e4m3fn`; se usa 240 como
margen conservador, no como limite del dtype. El loader custom esta en
`scripts/fp8_vibevoice.py`. El offload por modulos es opcional; este sprint no
implementa descarga circular del KV cache.


In [ ]:
# === Sprint 7.1: Conversion selectiva, validacion y persistencia FP8 ===
import gc, json, os, shutil, sys, time
from pathlib import Path

import jiwer
import librosa
import numpy as np
import soundfile as sf
import torch
import whisper

for _v in ["model", "model_fp8", "m_fp8", "wm", "out"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

S7_ROOT = Path.cwd()
if not (S7_ROOT / "weights").exists():
    S7_ROOT = S7_ROOT.parent
for path in [S7_ROOT, S7_ROOT / "VibeVoice_repo"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from scripts.fp8_vibevoice import (
    DynamicScaledFP8Linear,
    load_fp8_vibevoice,
    replace_qwen_linears_with_fp8,
)
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ES_PATH = S7_ROOT / "weights" / "vibevoice-1.5b-es-corrected"
FP8_OUT = S7_ROOT / "weights" / "vibevoice-1.5b-es-fp8"
SPRINT7_OUTPUT = S7_ROOT / "outputs" / "sprint7_fp8"
SPRINT7_OUTPUT.mkdir(parents=True, exist_ok=True)
TARGET_SR = 24000
FP8_RANGE_LIMIT = 240.0

assert torch.cuda.is_available(), "Sprint 7 requiere una GPU CUDA."
assert hasattr(torch, "_scaled_mm"), "Este PyTorch no expone torch._scaled_mm."
fp8_max = torch.finfo(torch.float8_e4m3fn).max
assert FP8_RANGE_LIMIT < fp8_max
print(f"GPU: {torch.cuda.get_device_name(0)} | FP8 max={fp8_max} | limite={FP8_RANGE_LIMIT}")

print("Cargando VibeVoice-ES en BF16 para conversion selectiva...")
model_fp8 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    str(MODEL_ES_PATH), torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"}, attn_implementation="sdpa",
)
model_fp8.eval()
model_fp8.set_ddpm_inference_steps(num_steps=20)
total_params_fp8 = sum(param.numel() for param in model_fp8.parameters())
fp8_params = sum(
    module.weight.numel()
    for name, module in model_fp8.named_modules()
    if name.startswith("model.language_model.") and isinstance(module, torch.nn.Linear)
)
converted_names_fp8 = replace_qwen_linears_with_fp8(
    model_fp8, range_limit=FP8_RANGE_LIMIT
)
fp8_modules = [
    module for module in model_fp8.modules()
    if isinstance(module, DynamicScaledFP8Linear)
]
assert len(converted_names_fp8) == len(fp8_modules) == 196
assert all(name.startswith("model.language_model.") for name in converted_names_fp8)
assert all(module.weight.dtype == torch.float8_e4m3fn for module in fp8_modules)
assert all(torch.isfinite(module.weight.float()).all() for module in fp8_modules)
assert all(module.weight.float().abs().max() <= FP8_RANGE_LIMIT for module in fp8_modules)
assert model_fp8.model.language_model.embed_tokens.weight.dtype == torch.bfloat16
assert model_fp8.lm_head.weight is model_fp8.model.language_model.embed_tokens.weight
for protected_name in [
    "prediction_head", "acoustic_tokenizer", "semantic_tokenizer",
    "acoustic_connector", "semantic_connector",
]:
    protected = getattr(model_fp8.model, protected_name)
    assert not any(isinstance(module, DynamicScaledFP8Linear) for module in protected.modules())

fp8_fraction = fp8_params / total_params_fp8
vram_fp8_idle = torch.cuda.memory_allocated() / 1024**3
print(f"Capas DynamicScaledFP8Linear: {len(fp8_modules)}")
print(f"Parametros FP8             : {fp8_params:,} ({fp8_fraction:.1%})")
print(f"VRAM reposo FP8 hibrido   : {vram_fp8_idle:.2f} GB")

model_fp8.config.vibevoice_fp8_config = {
    "format": "e4m3fn", "range_limit": FP8_RANGE_LIMIT,
    "activation_scaling": "dynamic_per_tensor",
    "weight_scaling": "static_per_layer",
    "compute_kernel": "torch._scaled_mm",
    "output_dtype": "bfloat16",
    "loader": "scripts.fp8_vibevoice.load_fp8_vibevoice",
}
if FP8_OUT.exists():
    shutil.rmtree(FP8_OUT)
model_fp8.save_pretrained(str(FP8_OUT), safe_serialization=True, max_shard_size="4GB")
processor_fp8 = VibeVoiceProcessor.from_pretrained(str(MODEL_ES_PATH))
processor_fp8.save_pretrained(str(FP8_OUT))
disk_fp8_gb = sum(path.stat().st_size for path in FP8_OUT.rglob("*.safetensors")) / 1024**3
print(f"Modelo FP8 guardado: {FP8_OUT} ({disk_fp8_gb:.2f} GB)")

del fp8_modules, model_fp8
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# === Sprint 7.2: Recarga del checkpoint y benchmark TTS ===
m_fp8 = load_fp8_vibevoice(FP8_OUT, device_map={"": "cuda:0"})
m_fp8.set_ddpm_inference_steps(num_steps=20)
reloaded_fp8_names = [
    name for name, module in m_fp8.named_modules()
    if isinstance(module, DynamicScaledFP8Linear)
]
assert reloaded_fp8_names == converted_names_fp8
assert m_fp8.model.language_model.embed_tokens.weight.dtype == torch.bfloat16
assert m_fp8.lm_head.weight is m_fp8.model.language_model.embed_tokens.weight
vram_fp8_idle = torch.cuda.memory_allocated() / 1024**3
print(f"Recarga FP8 verificada: {len(reloaded_fp8_names)} capas, {vram_fp8_idle:.2f} GB")

# En GPUs menores puede usarse device_map="auto" y max_memory. Esto ofrece
# offload por modulos de Accelerate, no descarga circular custom del KV cache.
test_texts_fp8 = [
    "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.",
    "La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.",
    "El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.",
    "Los modelos de lenguaje pueden generar texto y voz con gran precision.",
]
audios_fp8 = []
rtfs_fp8 = []
peak_vram_fp8 = vram_fp8_idle
for text in test_texts_fp8:
    inputs_fp8 = processor_fp8(
        text=[f"Speaker 1: {text}"], voice_samples=None, padding=True,
        return_tensors="pt", return_attention_mask=True,
    )
    inputs_fp8 = {
        key: (value.to(m_fp8.device) if torch.is_tensor(value) else value)
        for key, value in inputs_fp8.items() if value is not None
    }
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.inference_mode():
        out = m_fp8.generate(
            **inputs_fp8, max_new_tokens=600, tokenizer=processor_fp8.tokenizer,
            generation_config={"do_sample": False}, verbose=False,
        )
    elapsed = time.perf_counter() - start
    peak_vram_fp8 = max(peak_vram_fp8, torch.cuda.max_memory_allocated() / 1024**3)
    if out.speech_outputs and out.speech_outputs[0] is not None:
        audio = out.speech_outputs[0].squeeze().cpu().float().numpy()
        duration = len(audio) / TARGET_SR
        rtfs_fp8.append(elapsed / duration)
        audios_fp8.append(audio)
        sf.write(
            SPRINT7_OUTPUT / f"sample_{len(audios_fp8):02d}.wav",
            audio, TARGET_SR, subtype="PCM_16",
        )
        print(f"Audio: {duration:.1f}s | RTF: {rtfs_fp8[-1]:.2f}")
    else:
        audios_fp8.append(None)
        print("ERROR: generacion sin audio")

del m_fp8, out, inputs_fp8, reloaded_fp8_names
gc.collect()
torch.cuda.empty_cache()
vram_before_asr_fp8 = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM tras liberar VibeVoice: {vram_before_asr_fp8:.2f} GB")
if vram_before_asr_fp8 > 0.5:
    print("ADVERTENCIA: quedan referencias CUDA activas antes de Whisper.")


In [ ]:
# === Sprint 7.3: WER/CER con Whisper y reporte ===
whisper_cache = Path.home() / ".cache" / "whisper"
whisper_checkpoint = whisper_cache / "large-v3.pt"
if not whisper_checkpoint.exists():
    raise FileNotFoundError(
        f"Falta {whisper_checkpoint}. Descargue large-v3 por separado."
    )
print(f"Cargando Whisper large-v3 ({whisper_checkpoint.stat().st_size / 1024**3:.2f} GB)...")
asr_start = time.perf_counter()
wm = whisper.load_model("large-v3", device="cuda", download_root=str(whisper_cache))
print(f"Whisper cargado en {time.perf_counter() - asr_start:.1f}s")
wers_fp8 = []
cers_fp8 = []
for index, (text, audio) in enumerate(zip(test_texts_fp8, audios_fp8), start=1):
    if audio is None:
        wers_fp8.append(1.0)
        cers_fp8.append(1.0)
        continue
    print(f"Transcribiendo muestra {index}/{len(audios_fp8)}...", flush=True)
    audio_16k = librosa.resample(
        audio.astype(np.float32), orig_sr=TARGET_SR, target_sr=16000
    )
    transcript = wm.transcribe(
        audio_16k, language="es", fp16=True, verbose=False,
        condition_on_previous_text=False,
    )["text"].strip()
    wers_fp8.append(jiwer.wer(text.lower(), transcript.lower()))
    cers_fp8.append(jiwer.cer(text.lower(), transcript.lower()))
    print(f"[{index}] WER={wers_fp8[-1]:.3f} CER={cers_fp8[-1]:.3f} | {transcript[:80]}")

del wm
gc.collect()
torch.cuda.empty_cache()
rtf_fp8 = float(np.mean(rtfs_fp8)) if rtfs_fp8 else float("inf")
wer_fp8 = float(np.mean(wers_fp8)) if wers_fp8 else 1.0
cer_fp8 = float(np.mean(cers_fp8)) if cers_fp8 else 1.0
metrics_fp8 = {
    "model": "vibevoice-1.5b-es-fp8-dynamic",
    "format": "float8_e4m3fn",
    "range_limit": FP8_RANGE_LIMIT,
    "fp8_layers": len(converted_names_fp8),
    "fp8_parameter_fraction": fp8_fraction,
    "disk_gb": disk_fp8_gb,
    "vram_idle_gb": vram_fp8_idle,
    "vram_peak_gb": peak_vram_fp8,
    "rtf": rtf_fp8, "wer": wer_fp8, "cer": cer_fp8,
}
with open(SPRINT7_OUTPUT / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_fp8, file, indent=2)
print(
    f"\n=== FP8 DINAMICO === VRAM idle: {vram_fp8_idle:.2f} GB | "
    f"VRAM peak: {peak_vram_fp8:.2f} GB | RTF: {rtf_fp8:.4f} | "
    f"WER: {wer_fp8:.4f} | CER: {cer_fp8:.4f}"
)


---
# Sprint 8: Benchmark Final (7 Modelos + Estadistica)

> 200 muestras del split test de CV17.
> Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco.
> Shapiro-Wilk + Wilcoxon + Graficos + Tabla LaTeX.


In [ ]:
# === Sprint 8: Benchmark Final (placeholder - se completa al tener todos los modelos) ===
print("Sprint 8 se ejecuta despues de generar los 7 modelos.")
print("Modelos: FP16, IQ4_NL, GPTQ, AWQ, INT8, NF4, FP8")
print("200 muestras del split test de CV17")
print("Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco")
print("Stats: Shapiro-Wilk + Wilcoxon signed-rank")
print("Output: outputs/sprint8/ (CSV, graficos PNG, tabla LaTeX)")
